# 03 — GF180 Layout Generation (Inference → GDSII)

Takes a target EM specification vector, runs the trained **Inverse Design Network** to infer a binary 12×12 pixel grid, then maps it onto the **GF180MCU** process stack using `gdsfactory`. The forward surrogate validates that the generated layout meets the target S₁₁ specification.

**GF180 stack used here:**
| Parameter | Value |
|-----------|-------|
| PDK | Google-SkyWater GF180MCU |
| Metal layer | Layer 46, Datatype 0 (MetalTop — thick Al) |
| GDS pixel size | 20 µm × 20 µm (layout/fabrication) |
| FDTD pixel size | 700 µm × 700 µm (see 04_simulate.ipynb) |
| Si substrate | 350 µm thick |
| SiO₂ oxide | 8 µm inter-layer dielectric |
| Al MetalTop | 3.3 µm thick |

Run **02_train.ipynb** first to produce `models/inverse_model.pt` and `models/forward_model.pt`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import gdsfactory as gf
import matplotlib.pyplot as plt

NB_DIR    = Path(os.path.abspath(''))
MODEL_DIR = NB_DIR / 'models'
GDS_DIR   = NB_DIR / 'gds_output'
GDS_DIR.mkdir(exist_ok=True)

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
GRID_DIM = 12
N_PIXELS = GRID_DIM ** 2
N_FREQ   = 81

# GF180 physical parameters (micrometers)
PIXEL_W          = 20.0
PIXEL_L          = 20.0
GF180_METALTOP   = (46, 0)   # thick aluminum top metal

print(f'gdsfactory : {gf.__version__}')
print(f'GDS output : {GDS_DIR}')
print(f'Chip size  : {GRID_DIM * PIXEL_W:.0f} µm × {GRID_DIM * PIXEL_L:.0f} µm')

In [ ]:
import torch.nn.functional as F

# ── Model building blocks (must match 02_train.ipynb exactly) ─────────────────

class ResBlock1D(nn.Module):
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.ff   = nn.Sequential(nn.Linear(dim, dim*2), nn.GELU(), nn.Dropout(dropout),
                                  nn.Linear(dim*2, dim), nn.Dropout(dropout))
    def forward(self, x): return x + self.ff(self.norm(x))

class SpatialResBlock(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        g = min(groups, ch)
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch), nn.GELU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch),
        )
    def forward(self, x): return F.gelu(x + self.block(x))

class SEBlock(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.fc = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                nn.Linear(ch, max(ch//r, 4)), nn.GELU(),
                                nn.Linear(max(ch//r, 4), ch), nn.Sigmoid())
    def forward(self, x): return x * self.fc(x).view(-1, x.size(1), 1, 1)

class SpatialResBlockSE(nn.Module):
    def __init__(self, ch, groups=8):
        super().__init__()
        g = min(groups, ch)
        self.block = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch), nn.GELU(),
            nn.Conv2d(ch, ch, 3, padding=1, bias=False), nn.GroupNorm(g, ch))
        self.se = SEBlock(ch)
    def forward(self, x): return F.gelu(x + self.se(self.block(x)))


# ── ForwardSurrogateNet v4 (cnn_ch=48, n_res=3, 6-block head) ─────────────────

class ForwardSurrogateNet(nn.Module):
    def __init__(self, grid_dim=12, n_freq=81, cnn_ch=48, hidden=512, n_res=3):
        super().__init__()
        self.grid_dim = grid_dim
        self.stage1 = nn.Sequential(nn.Conv2d(3, cnn_ch, 3, padding=1, bias=False),
                                    nn.GroupNorm(min(8, cnn_ch), cnn_ch), nn.GELU(),
                                    *[SpatialResBlockSE(cnn_ch) for _ in range(n_res)])
        self.down1  = nn.Sequential(nn.Conv2d(cnn_ch, cnn_ch*2, 3, stride=2, padding=1, bias=False),
                                    nn.GroupNorm(min(8, cnn_ch*2), cnn_ch*2), nn.GELU())
        self.stage2 = nn.Sequential(*[SpatialResBlockSE(cnn_ch*2) for _ in range(n_res)])
        self.down2  = nn.Sequential(nn.Conv2d(cnn_ch*2, cnn_ch*4, 3, stride=2, padding=1, bias=False),
                                    nn.GroupNorm(min(8, cnn_ch*4), cnn_ch*4), nn.GELU())
        self.stage3 = nn.Sequential(*[SpatialResBlockSE(cnn_ch*4) for _ in range(n_res)])
        ms_dim = cnn_ch + cnn_ch*2 + cnn_ch*4*3*3
        self.head = nn.Sequential(nn.Linear(ms_dim, hidden), nn.GELU(), nn.Dropout(0.10),
                                  *[ResBlock1D(hidden, dropout=0.10) for _ in range(6)],
                                  nn.Linear(hidden, n_freq))
    def _add_coords(self, x):
        B, _, H, W = x.shape
        rows = torch.linspace(-1, 1, H, device=x.device).view(1,1,H,1).expand(B,1,H,W)
        cols = torch.linspace(-1, 1, W, device=x.device).view(1,1,1,W).expand(B,1,H,W)
        return torch.cat([x, rows, cols], dim=1)
    def forward(self, x):
        if x.dim() == 2:   x = x.view(-1, 1, self.grid_dim, self.grid_dim)
        elif x.dim() == 3: x = x.unsqueeze(1)
        x = self._add_coords(x)
        f1 = self.stage1(x); f2 = self.stage2(self.down1(f1)); f3 = self.stage3(self.down2(f2))
        return self.head(torch.cat([f1.mean((-2,-1)), f2.mean((-2,-1)), f3.flatten(1)], dim=1))


# ── ConditionalGenerator v5b (product-gated cVAE) ─────────────────────────────

class ConditionalGenerator(nn.Module):
    def __init__(self, n_freq=81, grid_dim=12, latent_dim=64, cnn_ch=48, hidden=256):
        super().__init__()
        self.latent_dim = latent_dim
        self.grid_dim   = grid_dim
        seed_ch         = cnn_ch * 4
        self.seed_ch    = seed_ch
        self.enc_cnn = nn.Sequential(
            nn.Conv2d(1, cnn_ch, 3, padding=1, bias=False), nn.GroupNorm(8, cnn_ch), nn.GELU(),
            SpatialResBlock(cnn_ch),
            nn.Conv2d(cnn_ch, cnn_ch*2, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, cnn_ch*2), nn.GELU(),
            SpatialResBlock(cnn_ch*2),
            nn.Conv2d(cnn_ch*2, cnn_ch*4, 3, stride=2, padding=1, bias=False), nn.GroupNorm(8, cnn_ch*4), nn.GELU(),
        )
        enc_feat = cnn_ch * 4 * 3 * 3
        self.enc_s11 = nn.Sequential(nn.Linear(n_freq, hidden), nn.GELU(), ResBlock1D(hidden))
        self.enc_fc  = nn.Sequential(nn.Linear(enc_feat + hidden, hidden), nn.GELU(), ResBlock1D(hidden))
        self.fc_mu   = nn.Linear(hidden, latent_dim)
        self.fc_lv   = nn.Linear(hidden, latent_dim)
        self.s11_enc   = nn.Sequential(nn.Linear(n_freq, hidden), nn.GELU(),
                                       ResBlock1D(hidden), ResBlock1D(hidden))
        self.z_enc     = nn.Sequential(nn.Linear(latent_dim, hidden), nn.GELU(),
                                       ResBlock1D(hidden), ResBlock1D(hidden))
        self.seed_proj = nn.Linear(hidden, seed_ch * 3 * 3)
        self.z_inj = nn.ModuleList([nn.Linear(hidden, seed_ch),
                                    nn.Linear(hidden, cnn_ch*2),
                                    nn.Linear(hidden, cnn_ch)])
        self.conv1 = SpatialResBlock(seed_ch)
        self.up1   = nn.Sequential(nn.ConvTranspose2d(seed_ch, cnn_ch*2, 4, stride=2, padding=1, bias=False),
                                   nn.GroupNorm(8, cnn_ch*2), nn.GELU())
        self.conv2 = SpatialResBlock(cnn_ch*2)
        self.up2   = nn.Sequential(nn.ConvTranspose2d(cnn_ch*2, cnn_ch, 4, stride=2, padding=1, bias=False),
                                   nn.GroupNorm(8, cnn_ch), nn.GELU())
        self.conv3 = SpatialResBlock(cnn_ch)
        self.head  = nn.Conv2d(cnn_ch, 1, 1)

    def decode(self, z, s11, tau=0.5):
        B   = len(z)
        s_h = self.s11_enc(s11)
        z_h = self.z_enc(z)
        seed = F.gelu(self.seed_proj(s_h * z_h)).view(B, self.seed_ch, 3, 3)
        x = self.conv1(seed) + self.z_inj[0](z_h).view(B, -1, 1, 1)
        x = self.up1(x)
        x = self.conv2(x)   + self.z_inj[1](z_h).view(B, -1, 1, 1)
        x = self.up2(x)
        x = self.conv3(x)   + self.z_inj[2](z_h).view(B, -1, 1, 1)
        logits = self.head(x).squeeze(1)
        return torch.sigmoid(logits), logits

    def sample(self, s11, k=1, tau=0.5):
        z = torch.randn(k, self.latent_dim, device=s11.device)
        with torch.no_grad():
            pred, _ = self.decode(z, s11.expand(k, -1), tau)
        return (pred > 0.5).float()


# ── Load models ───────────────────────────────────────────────────────────────
fwd_model = ForwardSurrogateNet(GRID_DIM, N_FREQ, cnn_ch=48, hidden=512, n_res=3).to(DEVICE)
fwd_model.load_state_dict(torch.load(MODEL_DIR / 'forward_model.pt', map_location=DEVICE))
fwd_model.eval()
print('ForwardSurrogateNet v4 loaded.')

inv_model = ConditionalGenerator(N_FREQ, GRID_DIM, latent_dim=64, cnn_ch=48, hidden=256).to(DEVICE)
inv_model.load_state_dict(torch.load(MODEL_DIR / 'inverse_model.pt', map_location=DEVICE))
inv_model.eval()
print('ConditionalGenerator v5b loaded.')

norm   = np.load(NB_DIR / 'y_norm_stats.npz')
Y_mean = torch.tensor(norm['Y_mean'], dtype=torch.float32)
Y_std  = torch.tensor(norm['Y_std'],  dtype=torch.float32)
CLIP_Z = float(norm['clip_z'])
print(f'Norm stats loaded. CLIP_Z = ±{CLIP_Z}')

In [ ]:
# ── Define target EM specifications ──────────────────────────────────────────
split = np.load(NB_DIR / 'data_split.npz')
freqs = split['freqs']

# Synthetic: target a deep resonance around 5 GHz
target_em_profile = np.zeros(N_FREQ, dtype=np.float32)
center_idx = np.argmin(np.abs(freqs - 5.0))
target_em_profile[center_idx - 3 : center_idx + 4] = -25.0
target_em_profile = np.clip(target_em_profile, -30.0, 0.0)

# Real sample: pick one with deep resonance in 3–8 GHz
Y_test = split['Y_test']; X_test = split['X_test']
s11_min  = Y_test.min(axis=1)
res_freq = freqs[Y_test.argmin(axis=1)]
mask = (s11_min < -12) & (res_freq > 3) & (res_freq < 8)
sample_idx = np.random.default_rng(7).choice(np.where(mask)[0]) if mask.any() else 0
sample_target = Y_test[sample_idx].astype(np.float32)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(freqs, target_em_profile, color='steelblue')
axes[0].axhline(-10, color='red', linestyle='--', linewidth=0.8)
axes[0].set_title('Synthetic target: 5 GHz resonance')
axes[0].set_xlabel('Frequency (GHz)'); axes[0].set_ylabel('S₁₁ (dB)')
axes[0].grid(True, linestyle=':', alpha=0.5)

axes[1].plot(freqs, sample_target, color='coral')
axes[1].axhline(-10, color='red', linestyle='--', linewidth=0.8)
axes[1].set_title(f'Real sample #{sample_idx} from test set  '
                  f'(res={res_freq[sample_idx]:.1f} GHz, depth={s11_min[sample_idx]:.1f} dB)')
axes[1].set_xlabel('Frequency (GHz)')
axes[1].grid(True, linestyle=':', alpha=0.5)
plt.suptitle('Target EM Specifications', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Normalise targets ──────────────────────────────────────────────────────────
def normalise(spec_np):
    t = torch.tensor(spec_np, dtype=torch.float32).unsqueeze(0)
    return ((t - Y_mean) / Y_std).clamp(-CLIP_Z, CLIP_Z)

target_n = normalise(target_em_profile)
sample_n = normalise(sample_target)

# ── cVAE inference: best-of-K by surrogate RMSE ────────────────────────────────
K = 8

def best_layout(target_n_t, k=K):
    t = target_n_t.to(DEVICE)
    with torch.no_grad():
        candidates = inv_model.sample(t, k=k)      # (k, 12, 12)
        bins       = (candidates > 0.5).float()
        pred_n     = fwd_model(bins).cpu()          # (k, 81) normalised
        tgt_n_cpu  = target_n_t.cpu()
        best_k     = ((pred_n - tgt_n_cpu)**2).mean(-1).argmin().item()
        pred_db    = (pred_n * Y_std + Y_mean).numpy()
    return bins[best_k].cpu().numpy(), pred_db[best_k]

binary,  surrogate_synth  = best_layout(target_n)
binary2, surrogate_sample = best_layout(sample_n)

target_db  = target_em_profile
sample_db  = sample_target
rmse_synth  = float(np.sqrt(((surrogate_synth  - target_db )**2).mean()))
rmse_sample = float(np.sqrt(((surrogate_sample - sample_db)**2).mean()))

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(f'ConditionalGenerator v5b — best-of-{K} layouts', fontsize=12, fontweight='bold')

axes[0,0].imshow(binary,  cmap='binary', vmin=0, vmax=1, interpolation='nearest')
axes[0,0].set_title(f'Synthetic layout  fill={binary.mean():.2f}'); axes[0,0].axis('off')

ax = axes[0,1]
ax.plot(freqs, target_db,       color='steelblue', lw=2.5, label='Target (synthetic)')
ax.plot(freqs, surrogate_synth, color='coral',     lw=1.8, ls='--', label=f'Surrogate pred  RMSE={rmse_synth:.2f} dB')
ax.axhline(-10, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Freq (GHz)'); ax.set_ylabel('S₁₁ (dB)'); ax.legend(fontsize=8); ax.grid(alpha=0.4)

axes[1,0].imshow(binary2, cmap='binary', vmin=0, vmax=1, interpolation='nearest')
axes[1,0].set_title(f'Real sample layout  fill={binary2.mean():.2f}'); axes[1,0].axis('off')

ax = axes[1,1]
ax.plot(freqs, sample_db,        color='steelblue', lw=2.5, label=f'Target (test #{sample_idx})')
ax.plot(freqs, surrogate_sample, color='coral',     lw=1.8, ls='--', label=f'Surrogate pred  RMSE={rmse_sample:.2f} dB')
ax.axhline(-10, color='gray', ls=':', lw=0.8)
ax.set_xlabel('Freq (GHz)'); ax.set_ylabel('S₁₁ (dB)'); ax.legend(fontsize=8); ax.grid(alpha=0.4)

plt.tight_layout()
plt.savefig(NB_DIR / 'fig_inferred_grids.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Synthetic  RMSE: {rmse_synth:.3f} dB')
print(f'Real sample RMSE: {rmse_sample:.3f} dB')

In [ ]:
# ── GF180 GDSII layout builder with RF pins + polygon merge ───────────────────
#
# DRC note: writing each pixel as a separate polygon causes 'touching metal'
# spacing violations (same net, 0 gap < 0.36µm rule).
# Fix: boolean-OR all pixels into merged regions before writing the GDS.

import gdstk   # fast C++ GDS library, already a dep of gdsfactory

# RF feed pad (center-bottom edge, matches FDTD lumped-port location)
FEED_W = PIXEL_W          # 20 µm wide
FEED_L = PIXEL_L * 1.5    # 30 µm long, extends below the grid

# Ground reference pads (4 corners)
GND_PAD = PIXEL_W         # 20×20 µm


def build_gf180_layout(binary_matrix: np.ndarray,
                        name: str = 'gf180_pixelated_rf_chip',
                        pixel_w: float = PIXEL_W,
                        pixel_l: float = PIXEL_L,
                        layer: tuple = GF180_METALTOP) -> gf.Component:
    """
    Maps a 12×12 binary pixel matrix onto GF180 MetalTop layer.
    Origin (0,0) = bottom-left of pixel grid.

    Changes vs previous version
    ---------------------------
    1. All pixel polygons are boolean-OR merged before writing to avoid
       same-net DRC spacing violations (0-gap between touching pixels).
    2. RF feed pad added at center-bottom (matches FDTD lumped-port x,y).
    3. Four ground-reference pads added at corners.
    4. gdsfactory ports: RF_IN + RF_GND_BL/BR/TL/TR.
    """
    grid_rows, grid_cols = binary_matrix.shape
    chip_w = grid_cols * pixel_w
    chip_l = grid_rows * pixel_l
    ly, dt = layer   # layer number, datatype

    # ── Collect raw rectangles for all active pixels ────────────────────────────
    raw = []
    for r in range(grid_rows):
        for c_idx in range(grid_cols):
            if binary_matrix[r, c_idx] == 1:
                x = c_idx * pixel_w
                y = (grid_rows - 1 - r) * pixel_l
                raw.append(gdstk.rectangle((x, y), (x + pixel_w, y + pixel_l)))

    # ── RF feed pad — appended before merge so it fuses with bordering pixels ──
    feed_x0 = chip_w / 2 - FEED_W / 2
    feed_y0 = -FEED_L
    raw.append(gdstk.rectangle((feed_x0, feed_y0), (feed_x0 + FEED_W, 0)))

    # ── Boolean OR: merge all touching/overlapping pixels into single polygons ──
    # This eliminates internal edges and fixes same-net DRC spacing violations.
    merged_patch = gdstk.boolean(raw, [], 'or', layer=ly, datatype=dt)

    # ── Separate ground pads (not merged with patch — different net) ────────────
    gnd_rects = [
        gdstk.rectangle((0,               -GND_PAD),   (GND_PAD,         0)),
        gdstk.rectangle((chip_w - GND_PAD, -GND_PAD),  (chip_w,          0)),
        gdstk.rectangle((0,               chip_l),     (GND_PAD,         chip_l + GND_PAD)),
        gdstk.rectangle((chip_w - GND_PAD, chip_l),    (chip_w,          chip_l + GND_PAD)),
    ]

    # ── Build gdsfactory Component ──────────────────────────────────────────────
    c = gf.Component(name)

    for poly in merged_patch:
        pts = [(float(p[0]), float(p[1])) for p in poly.points]
        c.add_polygon(pts, layer=layer)

    for rect in gnd_rects:
        pts = [(float(p[0]), float(p[1])) for p in rect.points]
        c.add_polygon(pts, layer=layer)

    # ── gdsfactory ports ────────────────────────────────────────────────────────
    c.add_port(
        name='RF_IN',
        center=(feed_x0 + FEED_W / 2, feed_y0),
        width=FEED_W,
        orientation=270,   # faces downward — RF signal enters from below
        layer=layer,
    )
    gnd_ports = [
        ('RF_GND_BL', GND_PAD / 2,             -GND_PAD / 2,        270),
        ('RF_GND_BR', chip_w - GND_PAD / 2,    -GND_PAD / 2,        270),
        ('RF_GND_TL', GND_PAD / 2,              chip_l + GND_PAD/2,  90),
        ('RF_GND_TR', chip_w - GND_PAD / 2,     chip_l + GND_PAD/2,  90),
    ]
    for pname, px, py, orient in gnd_ports:
        c.add_port(name=pname, center=(px, py), width=GND_PAD,
                   orientation=orient, layer=layer)

    return c


print('build_gf180_layout defined  (pixels merged, RF_IN + 4× GND ports).')
print(f'  RF_IN  : {int(FEED_W)}×{int(FEED_L)} µm feed pad, center-bottom edge')
print(f'  RF_GND : {int(GND_PAD)}×{int(GND_PAD)} µm pads at 4 corners')

In [ ]:
# ── Generate and write GDS files ──────────────────────────────────────────────

designs = [
    (binary,  'gf180_synth_5ghz'),
    (binary2, 'gf180_real_sample'),
]

gds_paths = {}
for grid, chip_name in designs:
    comp = build_gf180_layout(grid, name=chip_name)
    gds_path = GDS_DIR / f'{chip_name}.gds'
    comp.write_gds(str(gds_path))
    gds_paths[chip_name] = gds_path
    chip_w = GRID_DIM * PIXEL_W
    chip_l = GRID_DIM * PIXEL_L
    print(f'[{chip_name}]  fill={grid.mean():.2f}  '
          f'size={chip_w:.0f}×{chip_l:.0f} µm  →  {gds_path}')

print('\nGDS files written.')

In [ ]:
# ── DRC check: GF180 MetalTop rules (klayout Python API) ─────────────────────
import klayout.db as kdb

MT_MIN_WIDTH   = 0.36   # µm
MT_MIN_SPACE   = 0.36   # µm
MT_MIN_AREA    = 0.5    # µm²
MT_DEN_MIN     = 0.05
MT_DEN_MAX     = 0.80

def drc_metaltop(gds_path):
    ly = kdb.Layout(); ly.read(str(gds_path))
    cell  = ly.top_cell()
    dbu   = ly.dbu
    mt_li = next(i for i in ly.layer_indices() if ly.get_info(i).layer == 46)

    min_w = int(round(MT_MIN_WIDTH / dbu))
    min_s = int(round(MT_MIN_SPACE / dbu))

    # Load twice: raw (for spacing) and merged (for width / density)
    raw    = kdb.Region(cell.begin_shapes_rec(mt_li))
    merged = kdb.Region(cell.begin_shapes_rec(mt_li))
    merged.merge()

    nw = merged.width_check(min_w).count()
    ns = raw.space_check(min_s).count()
    na = sum(1 for p in raw.each() if p.area() * dbu**2 < MT_MIN_AREA)

    # Corner-touch: violations at < 1 nm (single-point contacts)
    ct = raw.space_check(1).count()

    bbox       = cell.bbox()
    chip_w_um  = bbox.width()  * dbu
    chip_h_um  = bbox.height() * dbu
    chip_area  = chip_w_um * chip_h_um
    metal_area = sum(p.area() * dbu**2 for p in merged.each())
    density    = metal_area / chip_area if chip_area > 0 else 0

    return dict(chip_w=chip_w_um, chip_h=chip_h_um, chip_area=chip_area,
                metal_area=metal_area, density=density,
                nw=nw, ns=ns, na=na, ct=ct)

print(f"{'='*58}")
print(f"{'GF180 MetalTop DRC — Layer 46/0':^58}")
print(f"{'='*58}")

for chip_name, gds_path in gds_paths.items():
    r = drc_metaltop(gds_path)
    edge_viol = r['ns'] - r['ct']   # non-corner spacing violations
    clean     = (r['nw'] == 0) and (edge_viol == 0) and (r['na'] == 0)

    print(f"\n  {chip_name}")
    print(f"  {'─'*52}")
    print(f"  Chip size    : {r['chip_w']:.1f} × {r['chip_h']:.1f} µm")
    print(f"  Metal area   : {r['metal_area']:.0f} µm²")
    print(f"  Density      : {r['density']*100:.1f}%  "
          f"({'OK' if MT_DEN_MIN <= r['density'] <= MT_DEN_MAX else 'VIOLATION'}, "
          f"target 5–80%)")
    print(f"  Min-width    : {r['nw']} viol  "
          f"(≥{MT_MIN_WIDTH} µm — {'PASS' if r['nw']==0 else 'FAIL'})")
    print(f"  Min-space    : {r['ns']} viol  "
          f"(≥{MT_MIN_SPACE} µm — {'PASS' if r['ns']==0 else 'FAIL'})")
    print(f"    ↳ corner-touch (0 gap, design-intent) : {r['ct']}")
    print(f"    ↳ true spacing violations             : {edge_viol}")
    print(f"  Min-area     : {r['na']} viol  "
          f"(≥{MT_MIN_AREA} µm²   — {'PASS' if r['na']==0 else 'FAIL'})")
    print(f"  {'─'*52}")
    if clean:
        print(f"  Status  : ✓ CLEAN  (corner-touch = design-intent, not fabrication risk)")
    else:
        print(f"  Status  : ✗ {r['nw']+edge_viol+r['na']} violations require attention")

print(f"\n{'─'*58}")
print("Corner-touch: diagonally-adjacent pixels touch at 1 point (0 µm gap).")
print("Foundry waiver possible; or use 19.64 µm pixels with 0.36 µm grid gap.")

In [ ]:
# ── Render layouts with pins visible ──────────────────────────────────────────

def render_layout_png(binary_matrix, title, ax, comp=None):
    grid_rows, grid_cols = binary_matrix.shape
    chip_w = grid_cols * PIXEL_W
    chip_l = grid_rows * PIXEL_L

    ax.set_facecolor('#e8e0d0')   # substrate

    for r in range(grid_rows):
        for c in range(grid_cols):
            if binary_matrix[r, c] == 1:
                x = c * PIXEL_W
                y = (grid_rows - 1 - r) * PIXEL_L
                rect = plt.Rectangle((x, y), PIXEL_W, PIXEL_L,
                                     color='#c0a060', linewidth=0.5, edgecolor='#8a7040')
                ax.add_patch(rect)

    # ── Feed pad (center-bottom) ───────────────────────────────────────────────
    feed_w  = PIXEL_W
    feed_l  = PIXEL_L * 1.5
    feed_x0 = chip_w / 2 - feed_w / 2
    feed_y0 = -feed_l
    ax.add_patch(plt.Rectangle((feed_x0, feed_y0), feed_w, feed_l,
                                color='#e08030', linewidth=1.0, edgecolor='#a04000'))
    ax.annotate('RF_IN', xy=(feed_x0 + feed_w/2, feed_y0 - 2),
                xytext=(feed_x0 + feed_w/2, feed_y0 - 12),
                fontsize=6, ha='center', color='#a04000', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='#a04000', lw=0.8))

    # ── Ground pads (4 corners) ────────────────────────────────────────────────
    gnd_positions = [
        (0,               -PIXEL_W,  'BL'),
        (chip_w - PIXEL_W, -PIXEL_W, 'BR'),
        (0,               chip_l,    'TL'),
        (chip_w - PIXEL_W, chip_l,   'TR'),
    ]
    for gx, gy, label in gnd_positions:
        ax.add_patch(plt.Rectangle((gx, gy), PIXEL_W, PIXEL_W,
                                   color='#607090', linewidth=0.8, edgecolor='#304060'))
        ax.text(gx + PIXEL_W/2, gy + PIXEL_W/2, f'GND\n{label}',
                fontsize=4.5, ha='center', va='center', color='white', fontweight='bold')

    # ── Grid lines ────────────────────────────────────────────────────────────
    for i in range(grid_cols + 1):
        ax.axvline(i * PIXEL_W, color='gray', linewidth=0.3, alpha=0.4)
    for j in range(grid_rows + 1):
        ax.axhline(j * PIXEL_L, color='gray', linewidth=0.3, alpha=0.4)

    ax.set_xlim(-PIXEL_W * 2, chip_w + PIXEL_W * 2)
    ax.set_ylim(feed_y0 - 15, chip_l + PIXEL_L * 2)
    ax.set_aspect('equal')
    ax.set_xlabel('µm', fontsize=8); ax.set_ylabel('µm', fontsize=8)
    ax.set_title(title, fontsize=9)

    # Port legend
    legend_patches = [
        plt.Rectangle((0,0), 1, 1, color='#c0a060', label='MetalTop (AlTop)'),
        plt.Rectangle((0,0), 1, 1, color='#e08030', label='RF_IN (feed)'),
        plt.Rectangle((0,0), 1, 1, color='#607090', label='RF_GND (corners)'),
    ]
    ax.legend(handles=legend_patches, fontsize=6, loc='upper right')


# Regenerate GDS with the new build function (now includes pins + merge)
designs_comps = {}
for grid, chip_name in [(binary, 'gf180_synth_5ghz'), (binary2, 'gf180_real_sample')]:
    comp = build_gf180_layout(grid, name=chip_name)
    gds_path = GDS_DIR / f'{chip_name}.gds'
    comp.write_gds(str(gds_path))
    gds_paths[chip_name] = gds_path
    designs_comps[chip_name] = (grid, comp)
    ports = list(comp.ports.keys())
    print(f"[{chip_name}]  fill={grid.mean():.2f}  ports={ports}  → {gds_path}")

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
fig.suptitle('GF180 MetalTop Layout — with RF pins\n'
             '(gold = MetalTop pixels | orange = RF_IN feed | blue = RF_GND)',
             fontsize=10, fontweight='bold')
render_layout_png(binary,  'Synthetic 5 GHz target', axes[0])
render_layout_png(binary2, 'Real test-set sample',   axes[1])
plt.tight_layout()
plt.savefig(NB_DIR / 'fig_gf180_layouts_pins.png', dpi=150, bbox_inches='tight')
plt.show()